# Le prix est une duale

*Sur un ordre de mérite à trois centrales, résolu à la main puis en programme linéaire puis dans pommes_craft, on voit pourquoi le multiplicateur de la contrainte d'équilibre est le prix de l'électricité, comment le lire dans pommes 0.11.0 et ce qu'il ne dit pas.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/prix-dual.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`), comme l'explique la page [Installer et vérifier](https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets. Une ligne « Attention » signale un autre noyau : changez-le (Kernel › Change Kernel).

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), None)  # la racine du dépôt des cas
if RACINE is None:
    raise FileNotFoundError(
        "dépôt pommes-case-studies introuvable : ce notebook a besoin de son code et de ses données. "
        "Clonez le dépôt, placez ce notebook dedans (à sa racine ou dans un de ses dossiers) et ouvrez-le avec "
        "le noyau du venv du dépôt (https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html). "
        "Sans rien installer : ouvrez-le sur Google Colab.")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")
    print("Le noyau n'est sans doute pas celui du venv du dépôt : Kernel › Change Kernel.")

## 1. Trois centrales, trois heures, à la main

Trois centrales alimentent une zone. Base : 10 GW à 15 €/MWh. Semi-base : 5 GW à 60 €/MWh. Pointe : 5 GW à 120 €/MWh. La demande vaut 8 GW à la première heure, 13 GW à la deuxième, 18 GW à la troisième.

On appelle les centrales par coût croissant. C'est l'ordre de mérite.

```text
heure  demande  base   semi-base  pointe  unité marginale  prix
h1      8 GW   8 GW    0 GW      0 GW    base              15 €/MWh
h2     13 GW  10 GW    3 GW      0 GW    semi-base         60 €/MWh
h3     18 GW  10 GW    5 GW      3 GW    pointe           120 €/MWh
```

Le prix de l'heure est le coût variable de la dernière centrale appelée, celle qui produirait le MWh de plus si la demande montait d'un cran.

Le coût de production se calcule heure par heure :

```text
h1 :  8 000 × 15                               = 120 000 €
h2 : 10 000 × 15 + 3 000 × 60                  = 330 000 €
h3 : 10 000 × 15 + 5 000 × 60 + 3 000 × 120    = 810 000 €
total                                          = 1 260 000 €
```

Les consommateurs paient 8 000 × 15 + 13 000 × 60 + 18 000 × 120 = 3 060 000 €. La différence, 1 800 000 €, est la rente inframarginale : 450 000 € à h2 et 1 050 000 € à h3 pour la base, 300 000 € à h3 pour la semi-base. Cette rente rembourse les investissements. La centrale marginale ne gagne rien.

## 2. Le même problème en programme linéaire

Variables : la production p_i,h de chaque centrale i à chaque heure h, et un délestage d_h au coût c_def. Objectif : minimiser la somme des c_i × p_i,h et des c_def × d_h. Contraintes : pour chaque heure, l'équilibre Σ_i p_i,h + d_h = D_h, avec un multiplicateur λ_h. Pour chaque centrale et chaque heure, p_i,h ≤ P_i, avec un multiplicateur μ_i,h ≥ 0. Enfin p_i,h ≥ 0.

Les conditions d'optimalité, dites KKT, tiennent en trois lignes pour une centrale à une heure donnée :

```text
centrale appelée mais non saturée (0 < p < P) :  c_i = λ_h          μ = 0
centrale saturée (p = P) :                       c_i = λ_h − μ_i,h  donc λ_h ≥ c_i
centrale non appelée (p = 0) :                   c_i ≥ λ_h
```

Le multiplicateur λ_h est le prix. Il vaut le coût de la centrale appelée mais non saturée : à h2, la semi-base, 60 €/MWh. Il dépasse le coût des centrales saturées et reste sous celui des centrales à l'arrêt. λ_h est aussi la dérivée du coût optimal par rapport à D_h : un MWh de plus à h2 coûte 60 €. Prix et valeur marginale de la demande sont le même nombre.

Le délestage se traite comme une centrale de coût c_def : il est non appelé tant que λ_h < c_def.

## 3. Le même problème dans pommes_craft

Script exécuté avec l'environnement épinglé du dépôt des cas (pommes 0.11.0, pommes_craft 0.6.0, linopy 0.10.1, HiGHS). Les capacités sont imposées par `power_capacity_investment_min = max`, sans coût d'investissement, comme dans `pommes_cases/systeme_fr.py`.

In [ ]:
import sys, tempfile
from pathlib import Path
import polars as pl
from pommes_craft import Area, ConversionTechnology, Demand, EconomicHypothesis, LoadShedding, TimeStepManager
from pommes_craft.core.model import EnergyModel
from pommes.utils import dual_scale
from pommes_cases.solve import build_lp, solve_with_extra_constraints

Y, H = 2030, [0, 1, 2]
def fixe(nom, mw, cout):  # capacité imposée, sans coût d'investissement
    return ConversionTechnology(name=nom, factor={"electricity": 1.0}, variable_cost=cout, life_span=1000,
                                power_capacity_investment_min=mw, power_capacity_investment_max=mw)
model = EnergyModel(name="trois_centrales", folder=Path(tempfile.mkdtemp()), hours=H, year_ops=[Y],
                    year_invs=[Y], year_decs=[Y + 1000], modes=[], resources=["electricity"])
with model.context():
    EconomicHypothesis("eco", discount_rate=0.0, year_ref=Y, planning_step=1)
    TimeStepManager("pas", time_step_duration=1.0, operation_year_duration=3.0)
    zone = Area("FR")
    zone.add_component(Demand(name="demande", resource="electricity",
                              demand=pl.DataFrame({"hour": H, "year_op": [Y] * 3, "demand": [8000., 13000., 18000.]})))
    zone.add_component(LoadShedding(name="defaillance", resource="electricity", cost=3000.))
    for nom, mw, cout in [("base", 10000., 15.), ("semi_base", 5000., 60.), ("pointe", 5000., 120.)]:
        zone.add_component(fixe(nom, mw, cout))
lp, params = build_lp(model)                 # le programme linéaire, avant résolution
lp = solve_with_extra_constraints(model)     # HiGHS
dual = lp.dual["operation_adequacy_constraint"].sel(area="FR", resource="electricity").squeeze()
print("coût total (EUR) :", float(lp.objective.value))
print("prix (EUR/MWh) :", (dual / dual_scale(params)).values.ravel())

Le script affiche d'abord une liste de paramètres manquants remplis par défaut (`spillage_cost: 0.0`, etc.), deux fois : `build_lp` et `solve_with_extra_constraints` construisent chacun le programme. Ce n'est pas une erreur. HiGHS n'affiche rien : son journal va dans un fichier. Les deux dernières lignes de la sortie :

```text
coût total (EUR) : 1260000.0
prix (EUR/MWh) : [ 15.  60. 120.]
```

Coût et prix sont ceux du calcul à la main. La contrainte d'équilibre s'appelle `operation_adequacy_constraint` dans pommes 0.11.0 (`pommes/model/build_model.py`). Sa duale se lit dans `lp.dual`, indexée par `area`, `resource`, `hour` et `year_op`, comme le fait `prix_horaires` dans `systeme_fr.py`.

Faut-il diviser par un facteur d'échelle ? Oui, en général. L'objectif de pommes pèse un euro d'exploitation de l'année y par `discount_factor × operation_year_weight × operation_year_duration / Σ time_step_duration`, et la duale porte le même poids. `pommes.utils.dual_scale(params)` renvoie ce facteur ; `dual / dual_scale` est un prix en €/MWh. Ici il vaut 1 : actualisation nulle, une année, `operation_year_duration` égale aux 3 heures modélisées. Avec `operation_year_duration=8760.0`, la duale brute devient 43 800, 175 200 et 350 400, le facteur vaut 2 920, la division redonne 15, 60 et 120. Sur 8 760 heures, comme dans `systeme_fr.py`, le facteur vaut 1. Le piège apparaît sur un modèle à peu d'heures ou à plusieurs années météo pondérées.

## 4. Ce que le prix dual ne dit pas

Rareté. Si la demande de h3 passe à 22 GW, les 20 GW de parc ne suffisent plus. Le modèle déleste 2 GW et le prix de h3 vaut 3 000 €/MWh, le coût du `LoadShedding`. Vérifié par exécution : coût total 7 500 000 €, prix 15, 60, 3 000. Sans composant `LoadShedding`, pommes_craft 0.6.0 remplit `load_shedding_cost = 0` et `max_capacity = nan`. Vérifié par exécution : le modèle déleste toute la demande aux trois heures, aucune centrale ne tourne, le coût total vaut 0 € et tous les prix tombent à zéro. Le `LoadShedding` à coût élevé n'est pas une option, c'est ce qui donne un sens au prix.

Rigidités absentes. Le programme linéaire ignore le minimum technique, le coût de démarrage, la montée en charge. Le prix du modèle est en paliers, un par coût variable ; le prix de marché est lissé par ces rigidités et par les stratégies d'offre. Les planches 2026 « Forme des prix et modélisation », remises en cours, comparent les deux.

Prévision parfaite. L'optimiseur connaît les trois heures d'avance. Le prix dual n'intègre ni l'incertitude de la veille pour le lendemain, ni le coût des réserves.

## 5. Une capacité a aussi une duale

La contrainte p_i,h ≤ P_i a son multiplicateur μ_i,h. Il vaut λ_h − c_i quand la centrale est saturée, zéro sinon. C'est la valeur d'un MW de plus à cette heure. Dans pommes, c'est la duale de `operation_conversion_power_max_constraint`, à diviser par le même facteur. Linopy rend la duale d'une contrainte ≤ avec un signe négatif en minimisation. Voici les valeurs obtenues sur le cas nominal, en valeur absolue :

```text
                 h1    h2     h3    somme
base              0    45    105    150 €/MW
semi_base         0     0     60     60 €/MW
pointe            0     0      0      0 €/MW
```

Un MW de base en plus économiserait 45 € à h2 et 105 € à h3, soit 150 € sur trois heures. Sur une année, cette somme est le revenu qu'un MW capte sur le marché. Si elle dépasse l'annuité de l'investissement, le modèle de planification investit ; sinon non.

Une interconnexion saturée est une capacité comme une autre : sa duale est l'écart de prix entre les deux zones, et le produit flux × écart de prix est la rente de congestion. C'est la question du cas F [Cas F — Que vaut une interconnexion France-Espagne ou France-Allemagne ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/F_interconnexion).

## Vous l'avez compris quand…

- Vous retrouvez à la main les prix 15, 60 et 120 €/MWh et le coût 1 260 000 €, puis les mêmes valeurs en sortie du script.
- Vous prédisez sans exécuter le prix de h3 pour une demande de 22 GW, puis le vérifiez : 3 000 €/MWh.
- Vous passez `operation_year_duration` de 3 à 8 760, lisez `dual_scale(params)` égal à 2 920, et expliquez pourquoi la duale brute change et le prix non.
- Vous retrouvez 150 €/MW pour la base en sommant la duale de `operation_conversion_power_max_constraint` sur les heures.

## Pour aller plus loin

- La contrainte d'adéquation et les contraintes de capacité en notation mathématique : Model description (2) [Model description (2) : adéquation, capacités et millésimes, stockage, carbone, imports, TURPE](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html). La page documente pommes 0.11.0 ; le nom `operation_adequacy_constraint` n'y figure pas, il vient du code.
- Lire `model.solution` et `model.dual` sur `test_case` : notebook d'exemple de pommes, à mettre à jour [Notebook d'exemple de pommes : résoudre test_case et lire solution et duales](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/notebooks/example_notebook.html).
- Le déroulé pommes_craft, de `EnergyModel` à `get_results()` : usage de pommes_craft, à mettre à jour [Usage de pommes_craft : créer, ajouter des composants, résoudre, lire, sauver](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes_craft/usage.html).
- La rente de congestion et la duale d'une capacité d'échange : le cas F, construit [Cas F — Que vaut une interconnexion France-Espagne ou France-Allemagne ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/F_interconnexion).

## Sources

1. [Dépôt pommes-case-studies (pommes_cases/solve.py, pommes_cases/systeme_fr.py)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)
2. [Code source de pommes (pommes/utils.py, dual_scale ; pommes/model/build_model.py, operation_adequacy_constraint)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes)
3. [Documentation pommes, Model description (adéquation, capacités ; page de la version 0.11.0, en notation mathématique)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html)
4. [README de pommes_craft (PyPI, version 0.6.0)](https://pypi.org/project/pommes-craft/)